In [ ]:
"""
All-in-one driver fatigue detection:
- Data collection with MediaPipe: EAR, MAR (yawn), head pose, temporal features
- Training: stacked ensemble of XGBoost, LightGBM, CatBoost
- Explainability: SHAP (now enabled for XGBoost base model only)
- Real-time GUI: Tkinter + OpenCV + alarm

Usage:
  1) Run script and choose option 1 to collect data to fatigue_features.csv
  2) Run script and choose option 2 to train stacked ensemble
  3) Run script and choose option 3 for real-time detection GUI

Note: Requires webcam and Windows for winsound beep (replace for other OS).
"""

import os
import math
import time
import threading

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, ClassifierMixin, clone

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import shap
import joblib
import matplotlib.pyplot as plt

import tkinter as tk
from tkinter import ttk

from PIL import Image, ImageTk
import winsound


# Global configuration

RANDOM_STATE = 42
DATA_CSV = "fatigue_features.csv"
MODEL_PKL = "fatigue_ensemble.pkl"
SHAP_PNG = "shap_summary.png"

BLINK_EAR_THRESH = 0.21
PROB_DROWSY_THRESH = 0.6
FRAME_WINDOW = 30
ALARM_MIN_FRAMES = 30  # frames of high probability before alarm

mp_face_mesh = mp.solutions.face_mesh

# Indices for eyes and mouth - approximate, adjust as needed based on MediaPipe FaceMesh docs
LEFT_EYE_IDX = [33, 160, 158, 133, 153, 144]
RIGHT_EYE_IDX = [362, 385, 387, 263, 373, 380]
MOUTH_IDX = [13, 14, 78, 308]  # top, bottom, left, right


# Geometry / feature functions

def euclidean_dist(a, b):
    return np.linalg.norm(a - b)


def eye_aspect_ratio(eye_pts):
    p1, p2, p3, p4, p5, p6 = eye_pts
    A = euclidean_dist(p2, p6)
    B = euclidean_dist(p3, p5)
    C = euclidean_dist(p1, p4)
    ear = (A + B) / (2.0 * C + 1e-6)
    return ear


def mouth_openness_ratio(mouth_pts):
    top, bottom, left, right = mouth_pts
    vertical = euclidean_dist(top, bottom)
    horizontal = euclidean_dist(left, right) + 1e-6
    return vertical / horizontal


def get_3d_face_points(image, landmarks, idxs):
    h, w = image.shape[:2]
    pts = []
    for i in idxs:
        lm = landmarks.landmark[i]
        pts.append([lm.x * w, lm.y * h, lm.z * w])
    return np.array(pts, dtype=np.float32)


def estimate_head_pose(image, face_landmarks):
    # Robust head pose; returns (0,0,0) if landmarks not usable
    idxs = [1, 199, 33, 263, 61, 291]  # nose, chin, eyes, mouth corners
    h, w = image.shape[:2]

    pts_2d = []
    for i in idxs:
        if i >= len(face_landmarks.landmark):
            return 0.0, 0.0, 0.0
        lm = face_landmarks.landmark[i]
        pts_2d.append([lm.x * w, lm.y * h])
    pts_2d = np.array(pts_2d, dtype=np.float32)
    if pts_2d.shape != (6, 2):
        return 0.0, 0.0, 0.0

    model_3d = np.array([
        [0.0, 0.0, 0.0],
        [0.0, -63.0, -12.0],
        [-43.0, 32.0, -26.0],
        [43.0, 32.0, -26.0],
        [-30.0, -28.0, -24.0],
        [30.0, -28.0, -24.0],
    ], dtype=np.float32)

    focal_length = w
    center = (w / 2, h / 2)
    cam_matrix = np.array([[focal_length, 0, center[0]],
                           [0, focal_length, center[1]],
                           [0, 0, 1]], dtype=np.float32)
    dist_coeffs = np.zeros((4, 1), dtype=np.float32)

    try:
        success, rot_vec, trans_vec = cv2.solvePnP(
            model_3d, pts_2d, cam_matrix, dist_coeffs, flags=cv2.SOLVEPNP_ITERATIVE
        )
        if not success:
            return 0.0, 0.0, 0.0
    except cv2.error:
        return 0.0, 0.0, 0.0

    rot_mat, _ = cv2.Rodrigues(rot_vec)
    sy = math.sqrt(rot_mat[0, 0] ** 2 + rot_mat[1, 0] ** 2)
    yaw = math.degrees(math.atan2(rot_mat[2, 1], rot_mat[2, 2]))
    pitch = math.degrees(math.atan2(-rot_mat[2, 0], sy))
    roll = math.degrees(math.atan2(rot_mat[1, 0], rot_mat[0, 0]))
    return yaw, pitch, roll


# Data collection

def collect_data(output_csv=DATA_CSV, seconds=120, blink_ear_thresh=BLINK_EAR_THRESH):
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Camera not opened")
        return

    face_mesh = mp_face_mesh.FaceMesh(refine_landmarks=True, max_num_faces=1)
    start_time = time.time()

    ear_window = []
    blink_window = []
    last_ear = None
    blink_count = 0

    current_label = 0  # 0 alert, 1 fatigued
    rows = []

    print("Data collection started.")
    print("Press 'f' to mark fatigued, 'a' for alert, 'q' to stop early.")
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = face_mesh.process(frame_rgb)

        ear = 0.0
        mar = 0.0
        yaw = pitch = roll = 0.0

        if res.multi_face_landmarks:
            lm = res.multi_face_landmarks[0]
            h, w = frame.shape[:2]

            left_eye = []
            right_eye = []
            for i in LEFT_EYE_IDX:
                lmk = lm.landmark[i]
                left_eye.append(np.array([lmk.x * w, lmk.y * h]))
            for i in RIGHT_EYE_IDX:
                lmk = lm.landmark[i]
                right_eye.append(np.array([lmk.x * w, lmk.y * h]))
            left_eye = np.array(left_eye)
            right_eye = np.array(right_eye)

            ear_left = eye_aspect_ratio(left_eye)
            ear_right = eye_aspect_ratio(right_eye)
            ear = (ear_left + ear_right) / 2.0

            mouth_pts = []
            for i in MOUTH_IDX:
                lmk = lm.landmark[i]
                mouth_pts.append(np.array([lmk.x * w, lmk.y * h]))
            mouth_pts = np.array(mouth_pts)
            mar = mouth_openness_ratio(mouth_pts)

            yaw, pitch, roll = estimate_head_pose(frame, lm)

            if last_ear is not None and last_ear > blink_ear_thresh and ear <= blink_ear_thresh:
                blink_count += 1
            last_ear = ear

        ear_window.append(ear)
        blink_window.append(blink_count)
        if len(ear_window) > FRAME_WINDOW:
            ear_window.pop(0)
            blink_window.pop(0)

        ear_mean = float(np.mean(ear_window)) if ear_window else 0.0
        ear_var = float(np.var(ear_window)) if ear_window else 0.0
        blinks_30 = (blink_window[-1] - blink_window[0]) if len(blink_window) > 1 else 0

        rows.append({
            "ear": ear,
            "mar": mar,
            "ear_mean_30": ear_mean,
            "ear_var_30": ear_var,
            "blinks_30": blinks_30,
            "yaw": yaw,
            "pitch": pitch,
            "roll": roll,
            "label": current_label
        })

        cv2.putText(frame, f"EAR: {ear:.3f}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(frame, f"MAR: {mar:.3f}", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(frame, f"Yaw:{yaw:.1f} Pitch:{pitch:.1f} Roll:{roll:.1f}",
                    (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        cv2.putText(frame, f"Label: {'Fatigued' if current_label == 1 else 'Alert'}",
                    (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

        cv2.imshow("Data Collection (f/a/q)", frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('f'):
            current_label = 1
        elif key == ord('a'):
            current_label = 0
        elif key == ord('q'):
            break

        if time.time() - start_time > seconds:
            print("Time limit reached.")
            break

    cap.release()
    cv2.destroyAllWindows()
    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f"Saved {len(df)} samples to {output_csv}")


# Stacked ensemble model

class StackedEnsemble(BaseEstimator, ClassifierMixin):
    def __init__(self, base_models=None, meta_model=None, n_folds=5):
        self.base_models = base_models
        self.meta_model = meta_model
        self.n_folds = n_folds

    def fit(self, X, y):
        self.base_models_ = [list() for _ in self.base_models]
        self.meta_model_ = clone(self.meta_model)
        kfold = StratifiedKFold(n_splits=self.n_folds, shuffle=True, random_state=RANDOM_STATE)

        oof_preds = np.zeros((X.shape[0], len(self.base_models)))
        for i, model in enumerate(self.base_models):
            for train_idx, hold_idx in kfold.split(X, y):
                # Skip folds where y is single-class (avoid CatBoost error)
                if np.unique(y[train_idx]).shape[0] < 2:
                    continue
                instance = clone(model)
                self.base_models_[i].append(instance)
                instance.fit(X[train_idx], y[train_idx])
                y_pred = instance.predict_proba(X[hold_idx])[:, 1]
                oof_preds[hold_idx, i] = y_pred
        self.meta_model_.fit(oof_preds, y)
        return self

    def predict_proba(self, X):
        meta_features = np.zeros((X.shape[0], len(self.base_models)))
        for i, models in enumerate(self.base_models_):
            if not models:
                # If a base model never trained (extreme imbalance), leave column as zeros
                continue
            preds = np.column_stack([m.predict_proba(X)[:, 1] for m in models])
            meta_features[:, i] = preds.mean(axis=1)
        return self.meta_model_.predict_proba(meta_features)

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


def train_model(csv_path=DATA_CSV, model_path=MODEL_PKL, shap_png=SHAP_PNG):
    if not os.path.exists(csv_path):
        print(f"{csv_path} not found. Collect data first.")
        return

    df = pd.read_csv(csv_path).dropna()
    if "label" not in df.columns:
        print("CSV must contain 'label' column.")
        return

    X = df.drop(columns=["label"]).values
    y = df["label"].values.astype(int)
    feature_names = df.columns.drop("label")

    unique_labels = np.unique(y)
    if unique_labels.shape[0] < 2:
        print("Error: dataset has only one class. Collect both alert (0) and fatigued (1) samples.")
        return

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )

    unique_train = np.unique(y_train)
    if unique_train.shape[0] < 2:
        print("Error: training split has only one class. Collect more balanced data.")
        return

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    base_models = [
        XGBClassifier(
            objective="binary:logistic",
            n_estimators=200,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            eval_metric="logloss",
            base_score=0.5,  # keep numeric to avoid base_score issues
            use_label_encoder=False
        ),
        LGBMClassifier(
            n_estimators=200,
            max_depth=-1,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        CatBoostClassifier(
            iterations=200,
            depth=4,
            learning_rate=0.05,
            loss_function="Logloss",
            verbose=False,
            random_state=RANDOM_STATE
        ),
    ]

    meta_model = LogisticRegression(max_iter=1000)

    ensemble = StackedEnsemble(base_models=base_models, meta_model=meta_model, n_folds=3)
    ensemble.fit(X_train_s, y_train)

    y_pred = ensemble.predict(X_test_s)
    y_proba = ensemble.predict_proba(X_test_s)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    print(f"Accuracy:  {acc:.3f}")
    print(f"Precision: {prec:.3f}")
    print(f"Recall:    {rec:.3f}")
    print(f"F1-score:  {f1:.3f}")

    # SHAP explainability step for the XGBoost base model
    try:
        # Use first fitted XGB model from the ensemble (first base model, first fold)
        xgb_models = ensemble.base_models_[0]
        if len(xgb_models) == 0:
            print("No XGBoost models fitted inside ensemble; skipping SHAP.")
        else:
            xgb_model = xgb_models[0]

            # TreeExplainer works directly with XGBClassifier in recent shap versions
            explainer = shap.TreeExplainer(xgb_model)
            shap_values = explainer.shap_values(X_train_s)

            plt.figure(figsize=(10, 6))
            shap.summary_plot(
                shap_values,
                X_train_s,
                feature_names=feature_names,
                show=False
            )
            plt.tight_layout()
            plt.savefig(shap_png, dpi=200)
            plt.close()
            print(f"Saved SHAP summary plot to {shap_png}")
    except Exception as e:
        # Robust to SHAP / XGBoost minor version issues
        print(f"SHAP explanation step failed: {e}")
        print("Model training completed, but SHAP plot not generated.")

    joblib.dump({"scaler": scaler, "ensemble": ensemble, "features": feature_names},
                model_path)
    print(f"Saved model to {model_path}")


# Real-time GUI application

class FatigueMonitorApp:
    def __init__(self, root, model_path=MODEL_PKL):
        self.root = root
        self.root.title("Driver Fatigue Detection")

        self.video_label = tk.Label(root)
        self.video_label.pack()

        self.status_var = tk.StringVar(value="Status: Initializing")
        ttk.Label(root, textvariable=self.status_var, font=("Arial", 14)).pack(pady=5)

        self.ear_var = tk.StringVar(value="EAR: 0.000")
        self.mar_var = tk.StringVar(value="MAR: 0.000")
        self.blink_var = tk.StringVar(value="Blinks30: 0")
        self.prob_var = tk.StringVar(value="Fatigue Prob: 0.00")

        info_frame = ttk.Frame(root)
        info_frame.pack(pady=5)
        ttk.Label(info_frame, textvariable=self.ear_var).grid(row=0, column=0, padx=5)
        ttk.Label(info_frame, textvariable=self.mar_var).grid(row=0, column=1, padx=5)
        ttk.Label(info_frame, textvariable=self.blink_var).grid(row=0, column=2, padx=5)
        ttk.Label(info_frame, textvariable=self.prob_var).grid(row=0, column=3, padx=5)

        self.cap = cv2.VideoCapture(0)
        if not self.cap.isOpened():
            self.status_var.set("Status: Camera error")
            self.running = False
            return

        if not os.path.exists(model_path):
            self.status_var.set("Status: Model not found, train first")
            self.running = False
            return

        bundle = joblib.load(model_path)
        self.scaler = bundle["scaler"]
        self.ensemble = bundle["ensemble"]
        self.feature_names = bundle["features"]

        self.face_mesh = mp_face_mesh.FaceMesh(refine_landmarks=True, max_num_faces=1)
        self.running = True

        self.ear_window = []
        self.blink_window = []
        self.last_ear = None
        self.blink_count = 0

        self.prob_window = []
        self.drowsy_frames = 0
        self.alarm_playing = False

        self.update_video()

    def play_alarm(self):
        try:
            winsound.Beep(2500, 700)
        except Exception:
            pass

    def stop(self):
        self.running = False
        if self.cap.isOpened():
            self.cap.release()
        cv2.destroyAllWindows()
        self.root.destroy()

    def update_video(self):
        if not self.running:
            return
        ret, frame = self.cap.read()
        if not ret:
            self.status_var.set("Status: Camera read error")
            self.root.after(10, self.update_video)
            return

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = self.face_mesh.process(frame_rgb)

        ear = 0.0
        mar = 0.0
        yaw = pitch = roll = 0.0

        if res.multi_face_landmarks:
            lm = res.multi_face_landmarks[0]
            h, w = frame.shape[:2]

            left_eye = []
            right_eye = []
            for i in LEFT_EYE_IDX:
                lmk = lm.landmark[i]
                left_eye.append(np.array([lmk.x * w, lmk.y * h]))
            for i in RIGHT_EYE_IDX:
                lmk = lm.landmark[i]
                right_eye.append(np.array([lmk.x * w, lmk.y * h]))
            left_eye = np.array(left_eye)
            right_eye = np.array(right_eye)

            ear_left = eye_aspect_ratio(left_eye)
            ear_right = eye_aspect_ratio(right_eye)
            ear = (ear_left + ear_right) / 2.0

            mouth_pts = []
            for i in MOUTH_IDX:
                lmk = lm.landmark[i]
                mouth_pts.append(np.array([lmk.x * w, lmk.y * h]))
            mouth_pts = np.array(mouth_pts)
            mar = mouth_openness_ratio(mouth_pts)

            yaw, pitch, roll = estimate_head_pose(frame, lm)

            if self.last_ear is not None and self.last_ear > BLINK_EAR_THRESH and ear <= BLINK_EAR_THRESH:
                self.blink_count += 1
            self.last_ear = ear

        self.ear_window.append(ear)
        self.blink_window.append(self.blink_count)
        if len(self.ear_window) > FRAME_WINDOW:
            self.ear_window.pop(0)
            self.blink_window.pop(0)

        ear_mean = float(np.mean(self.ear_window)) if self.ear_window else 0.0
        ear_var = float(np.var(self.ear_window)) if self.ear_window else 0.0
        blinks_30 = (self.blink_window[-1] - self.blink_window[0]) if len(self.blink_window) > 1 else 0

        feat_vec = np.array([[ear, mar, ear_mean, ear_var, blinks_30, yaw, pitch, roll]])
        feat_vec_s = self.scaler.transform(feat_vec)
        prob_fatigue = float(self.ensemble.predict_proba(feat_vec_s)[0, 1])

        self.prob_window.append(prob_fatigue)
        if len(self.prob_window) > FRAME_WINDOW:
            self.prob_window.pop(0)
        smooth_prob = float(np.mean(self.prob_window))

        if smooth_prob >= PROB_DROWSY_THRESH:
            self.drowsy_frames += 1
        else:
            self.drowsy_frames = 0

        drowsy = self.drowsy_frames >= ALARM_MIN_FRAMES

        if drowsy and not self.alarm_playing:
            self.alarm_playing = True
            threading.Thread(target=self.play_alarm, daemon=True).start()
        elif not drowsy:
            self.alarm_playing = False

        status_text = "Status: DROWSY" if drowsy else "Status: SAFE"
        color = (0, 0, 255) if drowsy else (0, 255, 0)
        cv2.putText(frame, status_text, (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
        cv2.putText(frame, f"Prob: {smooth_prob:.2f}", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        self.status_var.set(status_text)
        self.ear_var.set(f"EAR: {ear:.3f}")
        self.mar_var.set(f"MAR: {mar:.3f}")
        self.blink_var.set(f"Blinks30: {blinks_30}")
        self.prob_var.set(f"Fatigue Prob: {smooth_prob:.2f}")

        frame_rgb_disp = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = cv2.resize(frame_rgb_disp, (640, 480))
        im = Image.fromarray(img)
        imgtk = ImageTk.PhotoImage(image=im)
        self.video_label.imgtk = imgtk
        self.video_label.configure(image=imgtk)

        self.root.after(10, self.update_video)


def run_realtime_gui():
    root = tk.Tk()
    app = FatigueMonitorApp(root)
    root.protocol("WM_DELETE_WINDOW", app.stop)
    root.mainloop()


# Main menu

def main():
    print("Driver Fatigue Detection All-in-One")
    print("1) Collect data")
    print("2) Train ensemble")
    print("3) Run real-time GUI")
    choice = input("Enter choice (1/2/3): ").strip()

    if choice == "1":
        secs = input("Enter collection duration in seconds (default 120): ").strip()
        secs = int(secs) if secs.isdigit() else 120
        collect_data(seconds=secs)
    elif choice == "2":
        train_model()
    elif choice == "3":
        run_realtime_gui()
    else:
        print("Invalid choice.")


if __name__ == "__main__":
    main()


Driver Fatigue Detection All-in-One
1) Collect data
2) Train ensemble
3) Run real-time GUI


Enter choice (1/2/3):  3


C:\Users\ekamb\AppData\Local\Programs\Python\Python310\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
C:\Users\ekamb\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\ekamb\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\ekamb\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
 